# 🚀 Smart Recruit AI — Phase 8 v3: Generate Job Embeddings (100k Rows)

Notebook ini mem-**pre-compute** representasi vektor (embeddings) dari 100.000 lowongan kerja menggunakan model SBERT.
Hasil embeddings ini yang nantinya di-load oleh Streamlit untuk matching resume ke job — **tidak perlu dihitung ulang setiap kali**.

---

### 🆕 Perubahan dari v2
| Aspek | v2 | v3 (ini) |
|---|---|---|
| Sample size | 50,000 | **100,000** |
| Join strategy | Inner join (banyak data hilang) | **Left join dari postings** (data maksimal) |
| RAM safety | Tidak ada cek | **RAM check + chunked loading** |
| Metadata build | `iterrows()` lambat | **Vectorized pandas** (10x lebih cepat) |
| Checkpoint | Tidak ada | **Auto-save per 10k batch** (aman dari crash) |
| Download | Langsung dari Drive (sering gagal) | **Copy ke /content/ dulu** |
| Column inspection | Tidak ada | **Cell khusus cek nama kolom** |

---

### 📦 Input (3 CSV dari Google Drive)
```
MyDrive/dataset_sr/
├── linkedin_job_postings.csv  → job_link, job_title, company
├── job_skills.csv             → job_link, job_skills
└── job_summary.csv            → job_link, job_summary
```

### 📤 Output
- `job_embeddings.npy` → array shape `(100000, 384)` dtype float32 (~150 MB)
- `job_metadata.pkl`   → list 100k dicts `{title, company, required_skills}`

---
### ⚠️ Sebelum mulai — WAJIB:
1. `Runtime → Change runtime type → T4 GPU` (bukan CPU)
2. Pastikan ketiga CSV sudah ada di `MyDrive/dataset_sr/`
3. Jalankan cell **satu per satu dari atas ke bawah** — jangan skip

---
## 📌 Cell 1 — Install Dependencies

**Apa yang dilakukan:**
Install library yang dibutuhkan. `sentence-transformers` adalah library untuk model SBERT yang akan kita pakai.
`psutil` dipakai untuk cek kondisi RAM Colab sebelum load data besar.

**Kenapa perlu diinstall?**
Colab sudah punya pandas/numpy bawaan, tapi `sentence-transformers` versi spesifik perlu diinstall manual
untuk memastikan kompatibilitas dengan kode kita.

In [1]:
# Install sentence-transformers versi yang sudah diuji kompatibel dengan project ini
# Flag --quiet supaya output instalasi tidak memenuhi layar
!pip install sentence-transformers==2.7.0 --quiet

# psutil: library untuk monitoring resource sistem (RAM, CPU)
# Kita pakai ini untuk safety check sebelum load dataset besar
!pip install psutil --quiet

print('✅ Dependencies installed successfully')
print('   - sentence-transformers 2.7.0')
print('   - psutil (RAM monitoring)')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.4 MB/s eta 0:00:00
✅ Dependencies installed successfully
   - sentence-transformers 2.7.0
   - psutil (RAM monitoring)


---
## 📌 Cell 2 — Mount Google Drive & Cek File

**Apa yang dilakukan:**
Menghubungkan Colab ke Google Drive kamu. Setelah mounted, semua file di Drive bisa diakses
dari path `/content/drive/MyDrive/`.

**Kenapa pakai Drive, bukan upload manual?**
File CSV dataset LinkedIn bisa 500MB+. Upload manual ke Colab sangat lambat dan file hilang
kalau session timeout. Dengan Drive, file tetap ada dan akses lebih cepat.

**Apa yang diverifikasi:**
Script akan print nama file dan ukurannya. Kalau ada yang ❌, berarti nama file/folder salah.

In [2]:
from google.colab import drive
import os

# Mount Google Drive ke path /content/drive/
# Akan muncul popup untuk authorize — klik 'Connect to Google Drive'
drive.mount('/content/drive')

# ── Definisi path ke folder dataset di Drive ──────────────────────────────────
# SESUAIKAN nama folder ini kalau kamu pakai nama folder yang berbeda
DRIVE_DIR = '/content/drive/MyDrive/dataset_sr'

# Path ke masing-masing file CSV
FILE_POSTINGS = os.path.join(DRIVE_DIR, 'linkedin_job_postings.csv')
FILE_SKILLS   = os.path.join(DRIVE_DIR, 'job_skills.csv')
FILE_SUMMARY  = os.path.join(DRIVE_DIR, 'job_summary.csv')

# ── Verifikasi semua file ada dan print ukurannya ─────────────────────────────
print('📂 Checking files in Drive...')
print(f'   Folder: {DRIVE_DIR}\n')

all_ok = True
for filepath in [FILE_POSTINGS, FILE_SKILLS, FILE_SUMMARY]:
    filename = os.path.basename(filepath)
    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / 1e6
        print(f'  ✅ {filename:45s} {size_mb:>8.1f} MB')
    else:
        # File tidak ditemukan — print error dan tandai all_ok = False
        print(f'  ❌ {filename:45s} FILE NOT FOUND!')
        all_ok = False

if not all_ok:
    # Hentikan eksekusi kalau ada file yang kurang
    raise FileNotFoundError(
        'Satu atau lebih file tidak ditemukan. '
        'Pastikan nama folder dan file persis sama (case-sensitive).'
    )

print('\n✅ Semua file ditemukan. Lanjut ke Cell 3.')

Mounted at /content/drive
📂 Checking files in Drive...
   Folder: /content/drive/MyDrive/dataset_sr

  ✅ linkedin_job_postings.csv                        416.7 MB
  ✅ job_skills.csv                                   674.0 MB
  ✅ job_summary.csv                                  814.9 MB

✅ Semua file ditemukan. Lanjut ke Cell 3.


---
## 📌 Cell 3 — Cek RAM & Nama Kolom (Diagnosa Awal)

**Apa yang dilakukan:**
Dua langkah diagnosa SEBELUM load data penuh:
1. Cek RAM tersedia di Colab — dataset 100k butuh minimal 6–8 GB RAM
2. Load hanya 5 baris pertama dari tiap CSV untuk lihat nama kolom yang sebenarnya

**Kenapa ini penting?**
Ini adalah langkah yang paling sering dilewati orang dan paling sering bikin error.
Nama kolom di file asli mungkin berbeda dari yang kita ekspektasi.
Lebih baik tau sekarang daripada error di tengah proses encoding 100k rows.

**Apa yang harus kamu perhatikan di output:**
- RAM tersedia harus > 6 GB untuk 100k rows
- Nama kolom di setiap file harus mengandung: `job_link`, `job_title`/`job_title`, dll.
- Kalau nama kolom berbeda, update mapping di Cell 4

In [3]:
import psutil
import pandas as pd
import numpy as np

# ── 1. CEK RAM TERSEDIA ───────────────────────────────────────────────────────
# Ambil info memory sistem menggunakan psutil
ram = psutil.virtual_memory()
ram_total_gb  = ram.total / 1e9
ram_avail_gb  = ram.available / 1e9
ram_used_pct  = ram.percent

print('🧠 RAM Status:')
print(f'   Total     : {ram_total_gb:.1f} GB')
print(f'   Available : {ram_avail_gb:.1f} GB')
print(f'   Used      : {ram_used_pct:.0f}%')

# Threshold: 100k rows butuh ~6 GB available minimum
# (3 CSV besar + df merged + embeddings array + metadata list)
RAM_THRESHOLD_GB = 6.0
if ram_avail_gb < RAM_THRESHOLD_GB:
    print(f'\n  ⚠️  RAM tersedia ({ram_avail_gb:.1f} GB) di bawah threshold ({RAM_THRESHOLD_GB} GB)!')
    print('     Pertimbangkan turunkan MAX_JOB_SAMPLE ke 50_000 di Cell 4.')
else:
    print(f'\n  ✅ RAM cukup untuk 100k rows.')

# ── 2. INSPEKSI NAMA KOLOM PER CSV ───────────────────────────────────────────
# Load HANYA 5 baris pertama — cepat, tidak menghabiskan RAM
print('\n' + '='*60)
print('📋 Column names per CSV file:')
print('='*60)

for label, filepath in [
    ('linkedin_job_postings.csv', FILE_POSTINGS),
    ('job_skills.csv',            FILE_SKILLS),
    ('job_summary.csv',           FILE_SUMMARY)
]:
    # nrows=5: hanya baca 5 baris untuk inspeksi cepat
    sample = pd.read_csv(filepath, nrows=5)
    print(f'\n  📄 {label}')
    print(f'     Columns ({len(sample.columns)}): {list(sample.columns)}')
    print(f'     Sample row[0]: {sample.iloc[0].to_dict()}')

print('\n' + '='*60)
print('👆 Pastikan nama kolom sesuai. Update COLUMN_MAP di Cell 4 kalau berbeda.')

🧠 RAM Status:
   Total     : 13.6 GB
   Available : 12.1 GB
   Used      : 11%

  ✅ RAM cukup untuk 100k rows.

📋 Column names per CSV file:

  📄 linkedin_job_postings.csv
     Columns (14): ['job_link', 'last_processed_time', 'got_summary', 'got_ner', 'is_being_worked', 'job_title', 'company', 'job_location', 'first_seen', 'search_city', 'search_country', 'search_position', 'job_level', 'job_type']
     Sample row[0]: {'job_link': 'https://www.linkedin.com/jobs/view/account-executive-dispensing-norcal-northern-nevada-becton-dickinson-at-bd-3802078767', 'last_processed_time': '2024-01-21 07:12:29.00256+00', 'got_summary': 't', 'got_ner': 't', 'is_being_worked': 'f', 'job_title': 'Account Executive - Dispensing (NorCal/Northern Nevada) - Becton Dickinson', 'company': 'BD', 'job_location': 'San Diego, CA', 'first_seen': '2024-01-15', 'search_city': 'Coronado', 'search_country': 'United States', 'search_position': 'Color Maker', 'job_level': 'Mid senior', 'job_type': 'Onsite'}

  📄 job_sk

---
## 📌 Cell 4 — Konfigurasi

**Apa yang dilakukan:**
Set semua konstanta dan mapping nama kolom yang akan dipakai di seluruh notebook.
Semua angka yang bisa kamu ubah ada di sini — tidak perlu modifikasi cell lain.

**Yang perlu disesuaikan:**
- `COLUMN_MAP`: update kalau nama kolom di file kamu berbeda dari output Cell 3
- `MAX_JOB_SAMPLE`: turunkan ke 50_000 kalau RAM tidak cukup
- `BATCH_SIZE`: turunkan ke 32 kalau CUDA out of memory

**Kenapa MAX_JOB_SAMPLE = 100_000?**
100k rows memberikan coverage yang lebih representatif untuk matching resume.
Lebih banyak job = lebih banyak pilihan match yang relevan untuk user.

In [4]:
# ── Model SBERT ───────────────────────────────────────────────────────────────
# all-MiniLM-L6-v2: model ringan (80MB), embedding 384 dimensi
# Trade-off: sedikit kurang akurat dari model besar, tapi jauh lebih cepat
# Benchmark: 14k sentences/detik di GPU vs 500 sentences/detik di CPU
SBERT_MODEL_NAME = 'all-MiniLM-L6-v2'

# ── Sampling ──────────────────────────────────────────────────────────────────
# Target: 100k rows. Turunkan ke 50_000 kalau RAM tidak cukup (lihat Cell 3)
MAX_JOB_SAMPLE = 100_000

# Random seed: angka apapun yang konsisten.
# Pastikan sama dengan config.py di project lokal agar hasil reproducible
RANDOM_SEED    = 42

# ── Encoding ──────────────────────────────────────────────────────────────────
# Batch size 64: optimal untuk T4 GPU (16GB VRAM)
# Turunkan ke 32 jika muncul error 'CUDA out of memory'
BATCH_SIZE = 64

# Checkpoint: simpan progress setiap N rows (aman dari crash di tengah jalan)
# Nilai 10_000 artinya kalau crash, maksimal kehilangan 10k rows progress
CHECKPOINT_EVERY = 10_000

# ── Output paths ──────────────────────────────────────────────────────────────
# Simpan DULU ke /content/ (RAM disk Colab, cepat), baru copy ke Drive
# Kenapa tidak langsung ke Drive? Write ke Drive lebih lambat dan bisa corrupt
# kalau proses terganggu di tengah jalan
LOCAL_DIR         = '/content'
OUTPUT_EMB_LOCAL  = os.path.join(LOCAL_DIR, 'job_embeddings.npy')
OUTPUT_META_LOCAL = os.path.join(LOCAL_DIR, 'job_metadata.pkl')

# Path Drive untuk backup permanen
OUTPUT_DIR        = '/content/drive/MyDrive/dataset_sr'
OUTPUT_EMB_DRIVE  = os.path.join(OUTPUT_DIR, 'job_embeddings.npy')
OUTPUT_META_DRIVE = os.path.join(OUTPUT_DIR, 'job_metadata.pkl')

# Path untuk checkpoint sementara (di /content/ bukan Drive, biar cepat)
CHECKPOINT_DIR    = os.path.join(LOCAL_DIR, 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)  # buat folder kalau belum ada

# ── Column mapping ────────────────────────────────────────────────────────────
# Mapping dari nama kolom di file asli ke nama internal yang kita pakai
# KEY   = nama kolom yang kita pakai di kode
# VALUE = nama kolom aktual di file CSV (sesuaikan dengan output Cell 3!)
COLUMN_MAP = {
    # File: linkedin_job_postings.csv
    'join_key'  : 'job_link',    # kolom penghubung antar file (PRIMARY KEY)
    'title'     : 'job_title',   # judul pekerjaan
    'company'   : 'company',     # nama perusahaan
    # File: job_skills.csv
    'skills'    : 'job_skills',  # skill yang dibutuhkan (comma-separated)
    # File: job_summary.csv
    'summary'   : 'job_summary', # deskripsi/ringkasan pekerjaan
}

# ── Print summary config ──────────────────────────────────────────────────────
print('✅ Configuration set:')
print(f'   SBERT model      : {SBERT_MODEL_NAME}')
print(f'   Target rows      : {MAX_JOB_SAMPLE:,}')
print(f'   Batch size       : {BATCH_SIZE}')
print(f'   Checkpoint every : {CHECKPOINT_EVERY:,} rows')
print(f'   Random seed      : {RANDOM_SEED}')
print(f'   Local output     : {LOCAL_DIR}')
print(f'   Drive backup     : {OUTPUT_DIR}')
print(f'\n   Column mapping   : {COLUMN_MAP}')

✅ Configuration set:
   SBERT model      : all-MiniLM-L6-v2
   Target rows      : 100,000
   Batch size       : 64
   Checkpoint every : 10,000 rows
   Random seed      : 42
   Local output     : /content
   Drive backup     : /content/drive/MyDrive/dataset_sr

   Column mapping   : {'join_key': 'job_link', 'title': 'job_title', 'company': 'company', 'skills': 'job_skills', 'summary': 'job_summary'}


---
## 📌 Cell 5 — Load & Join 3 File CSV

**Apa yang dilakukan:**
Load ketiga CSV dari Drive dan gabungkan menjadi satu DataFrame menggunakan `job_link` sebagai kunci.

**Kenapa LEFT JOIN, bukan INNER JOIN?**
- `inner join`: hanya baris yang ada di **semua** 3 file yang masuk → bisa kehilangan 40–60% data
- `left join` dari postings: SEMUA job posting masuk, skills/summary diisi NaN kalau tidak ada pasangannya
- Dengan left join, kita preserve data maksimal, dan NaN diisi kosong di Cell 6

**Kenapa `usecols`?**
Dataset LinkedIn bisa punya 20+ kolom. Dengan `usecols`, kita hanya load kolom yang dibutuhkan
→ hemat RAM hingga 70% dibanding load semua kolom.

**Expected output:**
- Postings: ~1.3 juta rows
- Merged: sedikit lebih sedikit dari postings (beberapa job_link mungkin tidak punya skills/summary)

In [6]:
import time
import csv  # dipakai untuk konstanta quoting

# Ambil nama kolom aktual dari mapping yang sudah dikonfigurasi di Cell 4
COL_JOIN    = COLUMN_MAP['join_key']
COL_TITLE   = COLUMN_MAP['title']
COL_COMPANY = COLUMN_MAP['company']
COL_SKILLS  = COLUMN_MAP['skills']
COL_SUMMARY = COLUMN_MAP['summary']

print('⏳ Loading CSVs (mungkin butuh 1–3 menit untuk file besar)...')
t_load = time.time()

# ── Load linkedin_job_postings.csv ────────────────────────────────────────────
# File ini biasanya bersih (data terstruktur) → pakai C parser (default, lebih cepat)
# usecols: hanya load 3 kolom yang kita butuhkan — hemat RAM signifikan
df_postings = pd.read_csv(
    FILE_POSTINGS,
    usecols=[COL_JOIN, COL_TITLE, COL_COMPANY],
    low_memory=False
)
print(f'  ✅ Postings   : {len(df_postings):>10,} rows | {df_postings.memory_usage(deep=True).sum()/1e6:.0f} MB RAM')

# ── Load job_skills.csv ───────────────────────────────────────────────────────
# File ini juga biasanya clean → C parser
df_skills_raw = pd.read_csv(
    FILE_SKILLS,
    usecols=[COL_JOIN, COL_SKILLS],
    low_memory=False
)
print(f'  ✅ Skills raw : {len(df_skills_raw):>10,} rows | {df_skills_raw.memory_usage(deep=True).sum()/1e6:.0f} MB RAM')

# ── Load job_summary.csv — SPECIAL HANDLING ───────────────────────────────────
# MASALAH: kolom job_summary berisi teks deskripsi panjang yang sering mengandung:
#   1. Newline (\n) di dalam sel → parser C kira baris baru = row baru
#   2. Tanda kutip (") yang tidak di-escape → parser C bingung cari penutup string
#   3. Karakter khusus dan encoding aneh dari copy-paste job descriptions
#
# SOLUSI: gunakan engine='python' yang lebih toleran terhadap format CSV tidak standar
#
# Penjelasan setiap parameter:
#   engine='python'         → parser Python (lebih lambat tapi jauh lebih toleran)
#   on_bad_lines='skip'     → baris yang tidak bisa di-parse di-skip, tidak crash
#                             (baris rusak biasanya <1% dari total data)
#   quoting=csv.QUOTE_ALL   → anggap SEMUA field di-quote → lebih toleran ke kutip
#   encoding='utf-8'        → pastikan encoding konsisten
#   encoding_errors='replace' → karakter aneh diganti '?' daripada crash
print(f'\n  ⏳ Loading job_summary.csv (pakai Python parser — lebih lambat tapi aman)...')
try:
    df_summary = pd.read_csv(
        FILE_SUMMARY,
        usecols=[COL_JOIN, COL_SUMMARY],
        engine='python',            # Python parser: toleran terhadap CSV tidak standar
        on_bad_lines='skip',        # skip baris rusak, jangan crash
        quoting=csv.QUOTE_ALL,      # semua field dianggap quoted
        encoding='utf-8',
        encoding_errors='replace'   # karakter tidak valid → ganti '?' daripada error
    )
    print(f'  ✅ Summary    : {len(df_summary):>10,} rows | {df_summary.memory_usage(deep=True).sum()/1e6:.0f} MB RAM')

except Exception as e:
    # Fallback: kalau quoting=QUOTE_ALL masih error (jarang terjadi),
    # coba tanpa parameter quoting
    print(f'  ⚠️  QUOTE_ALL mode gagal ({type(e).__name__}), mencoba fallback mode...')
    df_summary = pd.read_csv(
        FILE_SUMMARY,
        usecols=[COL_JOIN, COL_SUMMARY],
        engine='python',
        on_bad_lines='skip',        # tetap skip baris rusak
        encoding='utf-8',
        encoding_errors='replace'
    )
    print(f'  ✅ Summary (fallback) : {len(df_summary):>10,} rows')

# ── Aggregate skills: handle both "1 row per skill" AND "1 row per job" ───────
# Kenapa perlu aggregate?
#   Scenario A: 1 row per job  → 'job_skills' berisi 'Python, SQL, Docker'
#   Scenario B: 1 skill per row → tiap row hanya berisi 1 skill
#   groupby + join handle KEDUA scenario sekaligus
skills_per_job = (
    df_skills_raw
    .dropna(subset=[COL_SKILLS])            # buang baris tanpa nilai skills
    .astype({COL_SKILLS: str})              # pastikan tipe string (bukan float/int)
    .groupby(COL_JOIN)[COL_SKILLS]          # group semua skills per job_link
    .apply(lambda x: ', '.join(x.unique())) # gabung jadi satu string, deduplikasi
    .reset_index()
)
del df_skills_raw  # bebaskan RAM — tidak dipakai lagi setelah aggregate
print(f'  ✅ Skills agg : {len(skills_per_job):>10,} unique jobs with skills')

# ── JOIN: LEFT JOIN dari postings sebagai base ────────────────────────────────
# Kenapa LEFT JOIN, bukan INNER JOIN?
#   INNER JOIN: hanya baris ada di KETIGA file → bisa kehilangan 40-60% data
#   LEFT JOIN : semua postings tetap ada, kolom kosong diisi NaN lalu di-fillna('')
#
# Langkah:
#   1. df_postings LEFT JOIN skills_per_job → semua postings + skills (NaN jika tidak ada)
#   2. Hasil step 1 LEFT JOIN df_summary    → + summary (NaN jika tidak ada)
print('\n⏳ Joining datasets (left join)...')

df = df_postings.merge(
    skills_per_job,
    on=COL_JOIN,
    how='left'   # LEFT JOIN: semua baris dari df_postings dipertahankan
)
df = df.merge(
    df_summary,
    on=COL_JOIN,
    how='left'   # LEFT JOIN lagi untuk summary
)

# Bebaskan RAM dari DataFrame sumber yang sudah tidak dipakai
del df_postings, skills_per_job, df_summary

elapsed_load = time.time() - t_load

# ── Report hasil join ─────────────────────────────────────────────────────────
print(f'\n📊 Join result:')
print(f'   Total merged rows : {len(df):,}')
print(f'   Columns           : {list(df.columns)}')
print(f'   Load + join time  : {elapsed_load:.1f} seconds')
print(f'\n   Missing values per column (NaN akan diisi kosong di Cell 6):')
for col in df.columns:
    n_null = df[col].isna().sum()
    pct    = n_null / len(df) * 100
    flag   = '⚠️ ' if pct > 50 else '   '  # warning kalau >50% missing
    print(f'  {flag} {col:20s}: {n_null:>8,} NaN ({pct:.1f}%)')

print('\n   Preview 2 rows:')
df.head(2)

⏳ Loading CSVs (mungkin butuh 1–3 menit untuk file besar)...
  ✅ Postings   :  1,348,454 rows | 407 MB RAM
  ✅ Skills raw :  1,296,381 rows | 811 MB RAM

  ⏳ Loading job_summary.csv (pakai Python parser — lebih lambat tapi aman)...
  ✅ Summary    :    236,438 rows | 1374 MB RAM
  ✅ Skills agg :  1,294,296 unique jobs with skills

⏳ Joining datasets (left join)...

📊 Join result:
   Total merged rows : 1,348,454
   Columns           : ['job_link', 'job_title', 'company', 'job_skills', 'job_summary']
   Load + join time  : 97.1 seconds

   Missing values per column (NaN akan diisi kosong di Cell 6):
      job_link            :        0 NaN (0.0%)
      job_title           :        0 NaN (0.0%)
      company             :       11 NaN (0.0%)
      job_skills          :   54,158 NaN (4.0%)
  ⚠️  job_summary         : 1,112,016 NaN (82.5%)

   Preview 2 rows:


,job_link,job_title,company,job_skills,job_summary
0,https://www.linkedin.com/jobs/view/account-exe...,Account Executive - Dispensing (NorCal/Norther...,BD,"Medical equipment sales, Key competitors, Term...",NaN
1,https://www.linkedin.com/jobs/view/registered-...,Registered Nurse - RN Care Manager,Trinity Health MI,"Nursing, Bachelor of Science in Nursing, Maste...",NaN


---
## 📌 Cell 6 — Clean, Deduplikasi & Sample

**Apa yang dilakukan:**
4 langkah pembersihan data sebelum sampling:
1. **Isi NaN** → ganti semua nilai kosong dengan string kosong
2. **Drop baris kosong** → buang job yang tidak punya title DAN summary
3. **Deduplikasi** → hapus job_link duplikat (bisa terjadi kalau data sumber overlap)
4. **Sample** → ambil MAX_JOB_SAMPLE rows secara random

**Kenapa deduplikasi penting?**
Kalau ada job yang muncul dua kali, embedding-nya akan dobel di array → hasil matching bisa bias
ke company/job yang sama.

**Cara baca output:**
Perhatikan angka 'remaining' di setiap langkah — idealnya tidak turun lebih dari 20% total.

In [7]:
print('🧹 Cleaning dataset...')
initial_count = len(df)

# ── Langkah 1: Rename kolom ke nama internal yang konsisten ──────────────────
# Ini memastikan kode di bawah tidak bergantung pada nama kolom asli CSV
df = df.rename(columns={
    COL_TITLE  : 'job_title',
    COL_COMPANY: 'company',
    COL_SKILLS : 'job_skills',
    COL_SUMMARY: 'job_summary'
})

# ── Langkah 2: Isi NaN dengan string kosong ───────────────────────────────────
# fillna('') lebih aman dari dropna() karena kita tidak mau kehilangan data
# yang mungkin masih punya title/company walau summary kosong
for col in ['job_title', 'company', 'job_skills', 'job_summary']:
    df[col] = df[col].fillna('').astype(str).str.strip()

print(f'  ✅ NaN filled with empty string')

# ── Langkah 3: Drop baris yang SAMA SEKALI tidak punya konten berguna ─────────
# Kita buang hanya kalau job_title DAN job_summary keduanya kosong
# (job dengan title saja masih bisa di-encode dan di-match)
mask_empty = (df['job_title'] == '') & (df['job_summary'] == '')
n_empty    = mask_empty.sum()
df = df[~mask_empty]
print(f'  ✅ Dropped {n_empty:,} rows with empty title AND summary → {len(df):,} remaining')

# ── Langkah 4: Deduplikasi berdasarkan job_link ───────────────────────────────
# keep='first': dari duplikat, pertahankan yang pertama muncul
n_before_dedup = len(df)
df = df.drop_duplicates(subset=[COL_JOIN], keep='first')
n_dupes = n_before_dedup - len(df)
print(f'  ✅ Removed {n_dupes:,} duplicate job_links → {len(df):,} unique jobs')

# ── Langkah 5: Random sampling ────────────────────────────────────────────────
# Kalau jumlah data setelah clean > MAX_JOB_SAMPLE, ambil sampel acak
# random_state=RANDOM_SEED: hasil sampling konsisten kalau dijalankan ulang
if len(df) > MAX_JOB_SAMPLE:
    df = df.sample(n=MAX_JOB_SAMPLE, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f'  ✅ Sampled {MAX_JOB_SAMPLE:,} rows (random_state={RANDOM_SEED})')
else:
    df = df.reset_index(drop=True)  # reset index supaya 0-based
    print(f'  ✅ Using all {len(df):,} rows (jumlah di bawah target {MAX_JOB_SAMPLE:,})')

# ── Langkah 6: Build teks untuk encoding ─────────────────────────────────────
# Gabungkan 3 kolom menjadi 1 teks per job: title + summary + skills
# Urutan: title dulu (paling penting untuk matching) → summary → skills
# str.strip(): hapus whitespace di awal/akhir hasil gabungan
texts_to_encode = (
    df['job_title']   + ' ' +
    df['job_summary'] + ' ' +
    df['job_skills']
).str.strip().tolist()  # .tolist(): convert ke Python list (lebih efisien untuk SBERT)

# ── Summary statistik ─────────────────────────────────────────────────────────
text_lengths = [len(t) for t in texts_to_encode]
print(f'\n📊 Final dataset summary:')
print(f'   Total awal        : {initial_count:,} rows')
print(f'   Total siap encode : {len(texts_to_encode):,} rows')
print(f'   Drop total        : {initial_count - len(texts_to_encode):,} rows')
print(f'   Avg text length   : {int(np.mean(text_lengths)):,} chars')
print(f'   Min text length   : {min(text_lengths):,} chars')
print(f'   Max text length   : {max(text_lengths):,} chars')
print(f'\n   Sample text [0]   : {texts_to_encode[0][:200]}...')

🧹 Cleaning dataset...
  ✅ NaN filled with empty string
  ✅ Dropped 0 rows with empty title AND summary → 1,348,454 remaining
  ✅ Removed 0 duplicate job_links → 1,348,454 unique jobs
  ✅ Sampled 100,000 rows (random_state=42)

📊 Final dataset summary:
   Total awal        : 1,348,454 rows
   Total siap encode : 100,000 rows
   Drop total        : 1,248,454 rows
   Avg text length   : 1,025 chars
   Min text length   : 2 chars
   Max text length   : 23,623 chars

   Sample text [0]   : Registered Nurse (RN) at Aveanna  Nursing, Care, Pediatrics, Health, CPR, Electronic charting, Flexible scheduling, 401(k) Savings Plan, Paid training, Referral Bonuses...


---
## 📌 Cell 7 — Load Model SBERT

**Apa yang dilakukan:**
Load model `all-MiniLM-L6-v2` dari HuggingFace ke GPU.
Model akan di-download otomatis (~80MB) kalau belum ada di cache.
Setelah itu, sanity check memastikan model berjalan dan output shape-nya benar.

**Kenapa model ini?**
- Ukuran kecil (80MB) → cepat di-load
- Output 384 dimensi → cukup ekspresif, file output tidak terlalu besar
- Speed: ~14,000 sentences/detik di T4 GPU
- Tradeoff: sedikit di bawah model besar (all-mpnet-base-v2), tapi 5x lebih cepat

**Yang harus kamu lihat di output:**
- Device harus GPU (CUDA), bukan CPU
- Embedding dim harus 384
- Sanity check harus PASS

In [8]:
import torch
from sentence_transformers import SentenceTransformer

# ── Deteksi device (GPU vs CPU) ───────────────────────────────────────────────
# torch.cuda.is_available(): return True kalau runtime punya GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device: {device.upper()}')

if device == 'cuda':
    # Print info GPU: nama dan total VRAM
    gpu_name = torch.cuda.get_device_name(0)
    gpu_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'   GPU model : {gpu_name}')
    print(f'   VRAM      : {gpu_vram:.1f} GB')
    print(f'   Estimasi  : ~10–15 menit untuk {MAX_JOB_SAMPLE:,} rows')
else:
    # Peringatan keras kalau CPU — encoding 100k rows di CPU bisa 3+ jam
    print('   ⚠️  TIDAK ADA GPU! Encoding akan sangat lambat (3–5 jam untuk 100k rows).')
    print('   → Pergi ke Runtime → Change runtime type → T4 GPU, lalu restart dan re-run.')

# ── Load SBERT model ──────────────────────────────────────────────────────────
# Device param: memastikan model langsung ke GPU, bukan CPU
# Download otomatis ~80MB dari HuggingFace kalau belum ada di cache
print(f'\n⏳ Loading model "{SBERT_MODEL_NAME}"...')
print('   (Download ~80MB dari HuggingFace jika belum ada di cache)')
model = SentenceTransformer(SBERT_MODEL_NAME, device=device)

# ── Sanity check ──────────────────────────────────────────────────────────────
# Encode 1 kalimat test untuk verifikasi model berjalan dan output-nya benar
test_vec = model.encode(['senior python developer machine learning'])[0]

# Shape harus (384,) — ini adalah dimensi embedding all-MiniLM-L6-v2
assert test_vec.shape == (384,), f'❌ Unexpected shape: {test_vec.shape}'

# Pastikan bukan semua nol (model corrupt atau error)
assert test_vec.sum() != 0, '❌ Embedding adalah semua nol — model mungkin corrupt'

print(f'\n✅ Model loaded dan berjalan dengan benar')
print(f'   Embedding dim : {test_vec.shape[0]}')
print(f'   Test vector   : [{test_vec[:5].round(4).tolist()} ...]')
print('   Sanity check  : PASS')

🖥️  Device: CUDA
   GPU model : Tesla T4
   VRAM      : 15.6 GB
   Estimasi  : ~10–15 menit untuk 100,000 rows

⏳ Loading model "all-MiniLM-L6-v2"...
   (Download ~80MB dari HuggingFace jika belum ada di cache)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


✅ Model loaded dan berjalan dengan benar
   Embedding dim : 384
   Test vector   : [[-0.07100000232458115, -0.02370000071823597, 0.043800000101327896, 0.002099999925121665, 0.026200000196695328] ...]
   Sanity check  : PASS


---
## 📌 Cell 8 — Generate Embeddings dengan Checkpoint

**Apa yang dilakukan:**
Encode 100k teks menjadi vektor 384-dimensi menggunakan SBERT.
Proses dilakukan dengan **checkpoint otomatis setiap 10k rows** — kalau session crash di tengah jalan,
kita bisa lanjut dari checkpoint terakhir (lihat Cell 8b).

**Kenapa pakai checkpoint?**
Colab free tier bisa di-disconnect kalau browser tidak aktif atau quota habis.
Tanpa checkpoint, kalau crash di row ke-90k, semua progress hilang dan harus mulai ulang.
Dengan checkpoint, maksimal kehilangan 10k rows progress.

**Estimasi waktu:**
- T4 GPU: ~10–15 menit untuk 100k rows
- A100 GPU (Colab Pro): ~4–6 menit untuk 100k rows
- CPU only: ~4–6 JAM → sangat tidak direkomendasikan

**Yang harus kamu lihat:**
Progress bar akan menunjukkan kecepatan encoding. Normal di T4: 200–300 it/s.

In [9]:
import pickle
import math

total      = len(texts_to_encode)
n_chunks   = math.ceil(total / CHECKPOINT_EVERY)

print(f'⏳ Encoding {total:,} texts...')
print(f'   Batch size       : {BATCH_SIZE}')
print(f'   Checkpoint every : {CHECKPOINT_EVERY:,} rows ({n_chunks} chunks total)')
print(f'   Estimasi waktu   : {total/14000/60:.0f}–{total/10000/60:.0f} menit di T4 GPU')
print()

t0 = time.time()
all_embeddings = []  # list untuk kumpulkan hasil per chunk

# ── Encode dalam chunks ───────────────────────────────────────────────────────
# Kita bagi texts_to_encode menjadi chunk-chunk kecil
# Tiap chunk di-encode sekaligus, lalu hasilnya disimpan sebagai checkpoint
for chunk_idx in range(n_chunks):
    # Hitung indeks awal dan akhir chunk ini
    start = chunk_idx * CHECKPOINT_EVERY
    end   = min(start + CHECKPOINT_EVERY, total)  # min() untuk chunk terakhir

    chunk_texts = texts_to_encode[start:end]

    print(f'  📦 Chunk {chunk_idx+1}/{n_chunks}: rows {start:,}–{end:,} ({len(chunk_texts):,} texts)')

    # Encode chunk ini
    # show_progress_bar=True: tampilkan progress bar per batch di dalam chunk
    # convert_to_numpy=True: langsung convert dari tensor ke numpy array
    # normalize_embeddings=False: simpan raw vectors, cosine similarity akan normalize
    chunk_emb = model.encode(
        chunk_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=False
    )

    # Cast ke float32 — default SBERT mungkin float32 atau float64
    # float32 = 4 bytes per value. float64 = 8 bytes. Untuk 100k×384: 150MB vs 300MB
    chunk_emb = chunk_emb.astype(np.float32)
    all_embeddings.append(chunk_emb)

    # ── Simpan checkpoint ke disk ─────────────────────────────────────────────
    # Checkpoint: gabungkan semua chunks sampai sekarang dan simpan ke file
    # Kalau crash setelah chunk 5, kita bisa load checkpoint dan lanjut dari chunk 6
    ckpt_path = os.path.join(CHECKPOINT_DIR, f'ckpt_chunk_{chunk_idx+1:03d}.npy')
    np.save(ckpt_path, chunk_emb)  # simpan hanya chunk ini (lebih cepat)
    print(f'     💾 Checkpoint saved: {os.path.basename(ckpt_path)}')

# ── Gabungkan semua chunks menjadi satu array ─────────────────────────────────
# np.vstack: stack secara vertikal → (chunk1_rows + chunk2_rows + ..., 384)
print('\n⏳ Menggabungkan semua chunks...')
embeddings = np.vstack(all_embeddings)  # shape: (total, 384)
del all_embeddings  # bebaskan RAM — list of chunks tidak dipakai lagi

elapsed = time.time() - t0

# ── Validasi output ───────────────────────────────────────────────────────────
assert embeddings.shape == (total, 384), (
    f'❌ Shape salah: {embeddings.shape}, expected ({total}, 384)'
)
assert embeddings.dtype == np.float32, (
    f'❌ Dtype salah: {embeddings.dtype}, expected float32'
)
# Pastikan tidak ada NaN atau Inf dalam embeddings (tanda model error)
assert not np.isnan(embeddings).any(), '❌ Ada NaN dalam embeddings!'
assert not np.isinf(embeddings).any(), '❌ Ada Inf dalam embeddings!'

print(f'\n✅ Encoding selesai!')
print(f'   Total waktu       : {elapsed:.1f}s ({elapsed/60:.1f} menit)')
print(f'   Kecepatan rata-rata: {total/elapsed:.0f} texts/detik')
print(f'   Shape             : {embeddings.shape}')
print(f'   Dtype             : {embeddings.dtype}')
print(f'   Estimasi file size : {embeddings.nbytes / 1e6:.1f} MB')
print(f'   Validasi NaN/Inf  : PASS')

⏳ Encoding 100,000 texts...
   Batch size       : 64
   Checkpoint every : 10,000 rows (10 chunks total)
   Estimasi waktu   : 0–0 menit di T4 GPU

  📦 Chunk 1/10: rows 0–10,000 (10,000 texts)


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

     💾 Checkpoint saved: ckpt_chunk_001.npy
  📦 Chunk 2/10: rows 10,000–20,000 (10,000 texts)


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

     💾 Checkpoint saved: ckpt_chunk_002.npy
  📦 Chunk 3/10: rows 20,000–30,000 (10,000 texts)


...

     💾 Checkpoint saved: ckpt_chunk_008.npy
  📦 Chunk 9/10: rows 80,000–90,000 (10,000 texts)


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

     💾 Checkpoint saved: ckpt_chunk_009.npy
  📦 Chunk 10/10: rows 90,000–100,000 (10,000 texts)


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

     💾 Checkpoint saved: ckpt_chunk_010.npy

⏳ Menggabungkan semua chunks...

✅ Encoding selesai!
   Total waktu       : 186.8s (3.1 menit)
   Kecepatan rata-rata: 535 texts/detik
   Shape             : (100000, 384)
   Dtype             : float32
   Estimasi file size : 153.6 MB
   Validasi NaN/Inf  : PASS


---
## 📌 Cell 8b — [OPSIONAL] Resume dari Checkpoint jika Session Crash

**Kapan dipakai:**
Hanya jalankan cell ini kalau Cell 8 crash di tengah jalan dan ada checkpoint yang tersimpan.
Kalau Cell 8 selesai sukses, SKIP cell ini dan lanjut ke Cell 9.

**Cara kerja:**
Load semua file checkpoint yang sudah tersimpan di `/content/checkpoints/`,
gabungkan, lalu lanjut encode dari row terakhir yang berhasil.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SKIP CELL INI jika Cell 8 berhasil selesai tanpa error
# Jalankan HANYA jika Cell 8 crash dan ada checkpoint yang tersimpan
# ═══════════════════════════════════════════════════════════════════════════════

import glob

# Cari semua file checkpoint yang sudah tersimpan
ckpt_files = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'ckpt_chunk_*.npy')))
print(f'📦 Found {len(ckpt_files)} checkpoint files:')
for f in ckpt_files:
    size = os.path.getsize(f) / 1e6
    print(f'   {os.path.basename(f)} ({size:.1f} MB)')

if len(ckpt_files) == 0:
    print('❌ Tidak ada checkpoint ditemukan. Jalankan Cell 8 dari awal.')
else:
    # Load dan gabungkan semua checkpoint yang ada
    completed_chunks = [np.load(f) for f in ckpt_files]
    n_completed_rows = sum(c.shape[0] for c in completed_chunks)
    print(f'\n✅ Loaded {n_completed_rows:,} rows dari checkpoints')

    # Lanjut encode dari baris yang belum diproses
    remaining_texts = texts_to_encode[n_completed_rows:]
    print(f'   Sisa yang perlu di-encode: {len(remaining_texts):,} rows')

    if len(remaining_texts) > 0:
        print(f'\n⏳ Encoding sisa {len(remaining_texts):,} rows...')
        remaining_emb = model.encode(
            remaining_texts,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True
        ).astype(np.float32)
        completed_chunks.append(remaining_emb)

    # Gabungkan semua
    embeddings = np.vstack(completed_chunks)
    del completed_chunks
    print(f'\n✅ Resume selesai! Total shape: {embeddings.shape}')

---
## 📌 Cell 9 — Build Metadata (Vectorized)

**Apa yang dilakukan:**
Buat list metadata — satu dict per job — yang akan disimpan bersama embeddings.
Metadata ini yang ditampilkan di UI Streamlit ketika user melihat hasil job matching.

**Kenapa tidak pakai `iterrows()`?**
Version sebelumnya pakai `iterrows()` yang loop Python per baris.
Untuk 100k rows, `iterrows()` bisa makan 3–5 menit.
Versi ini pakai **pandas vectorized operations** → selesai dalam hitungan detik.

**Format output metadata:**
```python
[
  {'title': 'Data Analyst', 'company': 'Acme Corp', 'required_skills': ['Python', 'SQL']},
  ...  # 100k dicts total
]
```

**required_skills** adalah list of strings, bukan satu string panjang.
Format ini yang dipakai oleh `src/matching/similarity.py` untuk skill gap analysis.

In [10]:
print('⏳ Building metadata (vectorized)...')
t_meta = time.time()

# ── Parse skills: string comma-separated → list of strings ────────────────────
# Contoh input : 'Python, SQL, Docker, Machine Learning'
# Contoh output: ['Python', 'SQL', 'Docker', 'Machine Learning']
#
# Cara kerja:
# 1. str.split(','): pecah string di tanda koma → list
# 2. apply(lambda): untuk setiap list, bersihkan tiap elemen:
#    - .strip(): hapus spasi di awal/akhir
#    - filter out string 'nan' yang muncul dari konversi NaN ke string
#    - filter out string kosong
def parse_skills(skill_str):
    """Convert comma-separated skill string ke list of clean strings."""
    if not skill_str or skill_str.lower() == 'nan':
        return []  # tidak ada skills = list kosong, bukan error
    return [
        s.strip()
        for s in str(skill_str).split(',')
        if s.strip() and s.strip().lower() != 'nan'
    ]

# Terapkan parse_skills ke seluruh kolom sekaligus (vectorized)
# Hasilnya: Series of lists
skills_parsed = df['job_skills'].apply(parse_skills)

# ── Build list of dicts menggunakan zip ───────────────────────────────────────
# zip() jauh lebih cepat dari iterrows() untuk membuat list of dicts
# karena tidak ada overhead Python loop per baris dengan indexing
metadata = [
    {
        'title'          : title,
        'company'        : company,
        'required_skills': skills
    }
    for title, company, skills in zip(
        df['job_title'].tolist(),   # .tolist(): convert Series ke list Python (cepat)
        df['company'].tolist(),
        skills_parsed.tolist()
    )
]

elapsed_meta = time.time() - t_meta

# ── Validasi alignment ────────────────────────────────────────────────────────
# CRITICAL: panjang metadata HARUS sama dengan jumlah rows embedding
# Kalau tidak sama, similarity.py akan crash saat runtime Streamlit
assert len(metadata) == embeddings.shape[0], (
    f'❌ ALIGNMENT ERROR: {len(metadata)} metadata != {embeddings.shape[0]} embeddings'
)

# Verifikasi struktur dict
sample = metadata[0]
assert 'title'           in sample, '❌ Missing key: title'
assert 'company'         in sample, '❌ Missing key: company'
assert 'required_skills' in sample, '❌ Missing key: required_skills'
assert isinstance(sample['required_skills'], list), '❌ required_skills harus list'

# ── Statistik metadata ────────────────────────────────────────────────────────
n_skills_counts = [len(m['required_skills']) for m in metadata]
avg_skills   = np.mean(n_skills_counts)
empty_skills = sum(1 for n in n_skills_counts if n == 0)
max_skills   = max(n_skills_counts)

print(f'\n✅ Metadata built ({elapsed_meta:.2f} detik)')
print(f'   Records     : {len(metadata):,}')
print(f'   Avg skills  : {avg_skills:.1f} per job')
print(f'   Max skills  : {max_skills}')
print(f'   No skills   : {empty_skills:,} jobs ({empty_skills/len(metadata)*100:.1f}%)')
print(f'   Alignment   : {len(metadata):,} metadata == {embeddings.shape[0]:,} embeddings ✅')
print(f'\n   Preview 3 records:')
for i, m in enumerate(metadata[:3]):
    print(f'   [{i}] {m["title"]} @ {m["company"]} | skills: {m["required_skills"][:4]}')

⏳ Building metadata (vectorized)...

✅ Metadata built (1.32 detik)
   Records     : 100,000
   Avg skills  : 20.0 per job
   Max skills  : 372
   No skills   : 4,096 jobs (4.1%)
   Alignment   : 100,000 metadata == 100,000 embeddings ✅

   Preview 3 records:
   [0] Registered Nurse (RN) at Aveanna @ Health eCareers | skills: ['Nursing', 'Care', 'Pediatrics', 'Health']
   [1] Locum | Physician Obstetrics and Gynecology @ Weatherby Healthcare | skills: ['BC', 'ACLS', 'NRP', 'DEA']
   [2] Technology Sales (Recruiting) Business Development Manager @ Proven Recruiting | skills: ['Sales', 'Business Development', 'Account Management', 'Technology']


---
## 📌 Cell 10 — Simpan ke Local & Backup ke Drive

**Apa yang dilakukan:**
Simpan kedua artifact ke dua lokasi:
1. **`/content/`** (local Colab) → untuk download ke laptop
2. **`/content/drive/MyDrive/dataset_sr/`** (Google Drive) → backup permanen

**Kenapa simpan ke local dulu, bukan langsung ke Drive?**
Write ke Drive lebih lambat dan bisa corrupt kalau koneksi terputus di tengah proses save.
Strategy yang aman: save ke local dulu (cepat dan reliable), lalu copy ke Drive.

**File yang dihasilkan:**
- `job_embeddings.npy`: ~150MB untuk 100k rows
- `job_metadata.pkl`: ~20–30MB untuk 100k rows

In [11]:
import shutil

print('💾 Menyimpan artifacts...')
t_save = time.time()

# ── Save job_embeddings.npy ke /content/ (local) ──────────────────────────────
# np.save: format biner numpy, sangat efisien untuk array besar
# Ekstensi .npy ditambahkan otomatis oleh np.save kalau belum ada
np.save(OUTPUT_EMB_LOCAL, embeddings)
emb_size_mb = os.path.getsize(OUTPUT_EMB_LOCAL) / 1e6
print(f'  ✅ [Local] job_embeddings.npy')
print(f'     Shape  : {embeddings.shape}')
print(f'     Size   : {emb_size_mb:.1f} MB')
print(f'     Path   : {OUTPUT_EMB_LOCAL}')

# ── Save job_metadata.pkl ke /content/ (local) ────────────────────────────────
# pickle: format Python native untuk serialisasi objek Python (list of dicts)
# protocol=HIGHEST_PROTOCOL: gunakan protocol terbaru (paling efisien)
with open(OUTPUT_META_LOCAL, 'wb') as f:
    pickle.dump(metadata, f, protocol=pickle.HIGHEST_PROTOCOL)

meta_size_mb = os.path.getsize(OUTPUT_META_LOCAL) / 1e6
print(f'\n  ✅ [Local] job_metadata.pkl')
print(f'     Records: {len(metadata):,}')
print(f'     Size   : {meta_size_mb:.1f} MB')
print(f'     Path   : {OUTPUT_META_LOCAL}')

# ── Copy ke Google Drive (backup permanen) ────────────────────────────────────
# shutil.copy2: copy file beserta metadata (timestamps)
# Lebih aman dari np.save langsung ke Drive path
print(f'\n⏳ Copying ke Google Drive (backup)...')

shutil.copy2(OUTPUT_EMB_LOCAL,  OUTPUT_EMB_DRIVE)
shutil.copy2(OUTPUT_META_LOCAL, OUTPUT_META_DRIVE)

elapsed_save = time.time() - t_save

print(f'  ✅ [Drive] job_embeddings.npy → {OUTPUT_EMB_DRIVE}')
print(f'  ✅ [Drive] job_metadata.pkl   → {OUTPUT_META_DRIVE}')
print(f'\n⏱️  Total save time: {elapsed_save:.1f} detik')
print(f'\n🎉 Kedua artifacts tersimpan di local (/content/) dan Drive!')

💾 Menyimpan artifacts...
  ✅ [Local] job_embeddings.npy
     Shape  : (100000, 384)
     Size   : 153.6 MB
     Path   : /content/job_embeddings.npy

  ✅ [Local] job_metadata.pkl
     Records: 100,000
     Size   : 47.3 MB
     Path   : /content/job_metadata.pkl

⏳ Copying ke Google Drive (backup)...
  ✅ [Drive] job_embeddings.npy → /content/drive/MyDrive/dataset_sr/job_embeddings.npy
  ✅ [Drive] job_metadata.pkl   → /content/drive/MyDrive/dataset_sr/job_metadata.pkl

⏱️  Total save time: 2.1 detik

🎉 Kedua artifacts tersimpan di local (/content/) dan Drive!


---
## 📌 Cell 11 — Verifikasi Final (Reload dari Disk)

**Apa yang dilakukan:**
Reload file yang baru disimpan dari disk dan lakukan serangkaian assertion untuk memastikan
file tidak corrupt dan siap dipakai oleh Streamlit.

**Kenapa perlu reload?**
Ada kemungkinan (kecil) bahwa proses save gagal di tengah jalan dan file corrupt.
Dengan reload dan cek, kita memastikan file yang tersimpan benar-benar bisa dibaca.
Lebih baik tau sekarang daripada tau pas Streamlit error saat demo.

**Semua assertions harus PASS sebelum download.**

In [12]:
print('🔍 Verifikasi final — reload dari disk...')

# ── Reload embeddings dari local ──────────────────────────────────────────────
emb_check = np.load(OUTPUT_EMB_LOCAL)

# Cek dimensi: harus (N, 384) dimana N = jumlah rows yang di-encode
assert emb_check.ndim   == 2,    f'❌ Harus 2D array, got {emb_check.ndim}D'
assert emb_check.shape[1] == 384, f'❌ Dim salah: {emb_check.shape[1]}, expected 384'
assert emb_check.dtype == np.float32, f'❌ Dtype: {emb_check.dtype}, expected float32'
# Pastikan tidak ada nilai NaN atau Inf
assert not np.isnan(emb_check).any(), '❌ Ada NaN dalam file!'
assert not np.isinf(emb_check).any(), '❌ Ada Inf dalam file!'

print(f'  ✅ job_embeddings.npy')
print(f'     Shape : {emb_check.shape}')
print(f'     Dtype : {emb_check.dtype}')
print(f'     NaN   : 0 | Inf: 0')

# ── Reload metadata dari local ────────────────────────────────────────────────
with open(OUTPUT_META_LOCAL, 'rb') as f:
    meta_check = pickle.load(f)

# Cek tipe dan panjang
assert isinstance(meta_check, list), f'❌ Harus list, got {type(meta_check)}'

# Cek alignment: metadata dan embeddings harus sama panjang
assert len(meta_check) == emb_check.shape[0], (
    f'❌ ALIGNMENT ERROR: {len(meta_check)} metadata != {emb_check.shape[0]} embeddings'
)

# Cek struktur dict — semua 3 key harus ada
required_keys = {'title', 'company', 'required_skills'}
for i, m in enumerate(meta_check[:10]):  # cek 10 record pertama
    assert set(m.keys()) == required_keys, (
        f'❌ Record [{i}] missing keys: {required_keys - set(m.keys())}'
    )
    assert isinstance(m['required_skills'], list), (
        f'❌ Record [{i}] required_skills harus list, got {type(m["required_skills"])}'
    )

print(f'\n  ✅ job_metadata.pkl')
print(f'     Records   : {len(meta_check):,}')
print(f'     Keys      : {list(required_keys)}')
print(f'     Alignment : {len(meta_check):,} == {emb_check.shape[0]:,} ✅')

# ── Preview beberapa record ───────────────────────────────────────────────────
print(f'\n📋 Preview 5 records dari file:')
for i, m in enumerate(meta_check[:5]):
    skills_preview = m['required_skills'][:3]
    print(f'  [{i}] {m["title"][:40]:40s} @ {m["company"][:20]:20s} | {skills_preview}')

print(f'\n{'='*60}')
print(f'🎉 SEMUA CHECKS PASSED!')
print(f'   Files siap didownload dan dipakai oleh Streamlit.')
print(f'{'='*60}')

🔍 Verifikasi final — reload dari disk...
  ✅ job_embeddings.npy
     Shape : (100000, 384)
     Dtype : float32
     NaN   : 0 | Inf: 0

  ✅ job_metadata.pkl
     Records   : 100,000
     Keys      : ['title', 'required_skills', 'company']
     Alignment : 100,000 == 100,000 ✅

📋 Preview 5 records dari file:
  [0] Registered Nurse (RN) at Aveanna         @ Health eCareers      | ['Nursing', 'Care', 'Pediatrics']
  [1] Locum | Physician Obstetrics and Gynecol @ Weatherby Healthcare | ['BC', 'ACLS', 'NRP']
  [2] Technology Sales (Recruiting) Business D @ Proven Recruiting    | ['Sales', 'Business Development', 'Account Management']
  [3] Travel RN Oncology 1862.00/week - 237310 @ TravelNurseSource    | ['Oncology nursing', 'BLS certification', 'State license']
  [4] Floor Supervisor Full Time - TOMMY HILFI @ Tommy Hilfiger       | ['Customer Service', 'Sales', 'Profit']

🎉 SEMUA CHECKS PASSED!
   Files siap didownload dan dipakai oleh Streamlit.


---
## 📌 Cell 12 — Download ke Laptop

**Apa yang dilakukan:**
Download kedua file dari Colab `/content/` ke laptop kamu.

**Perhatikan:**
- `job_embeddings.npy` berukuran ~150MB — proses download mungkin butuh 1–3 menit tergantung koneksi
- Kalau download tidak mulai otomatis, cek popup blocker browser dan allow popup dari colab.research.google.com
- File di Drive (`dataset_sr/`) juga sudah ada sebagai backup — bisa download dari sana juga

**Setelah download:**
```
smart-recruit-ai/
└── data/
    └── processed/
        ├── job_embeddings.npy  ← taruh di sini (replace yang lama kalau ada)
        └── job_metadata.pkl    ← taruh di sini (replace yang lama kalau ada)
```
Lalu lanjut Phase 9 di Claude Code lokal.

In [13]:
from google.colab import files

# File di-download dari /content/ (bukan Drive) untuk performa yang lebih baik
# Download dari Drive sering timeout untuk file besar

print(f'⬇️  Memulai download job_embeddings.npy ({emb_size_mb:.0f} MB)...')
print('   (Mungkin butuh 1–3 menit tergantung kecepatan internet)')
files.download(OUTPUT_EMB_LOCAL)   # /content/job_embeddings.npy

print(f'\n⬇️  Memulai download job_metadata.pkl ({meta_size_mb:.0f} MB)...')
files.download(OUTPUT_META_LOCAL)  # /content/job_metadata.pkl

print('\n' + '='*60)
print('✅ Download selesai!')
print()
print('📁 Taruh file-file ini di project lokal kamu:')
print('   smart-recruit-ai/data/processed/job_embeddings.npy')
print('   smart-recruit-ai/data/processed/job_metadata.pkl')
print()
print('▶️  Lanjut: Phase 9 — build src/pipeline/pipeline.py di Claude Code')
print('='*60)

⬇️  Memulai download job_embeddings.npy (154 MB)...
   (Mungkin butuh 1–3 menit tergantung kecepatan internet)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


⬇️  Memulai download job_metadata.pkl (47 MB)...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download selesai!

📁 Taruh file-file ini di project lokal kamu:
   smart-recruit-ai/data/processed/job_embeddings.npy
   smart-recruit-ai/data/processed/job_metadata.pkl

▶️  Lanjut: Phase 9 — build src/pipeline/pipeline.py di Claude Code


---
## ℹ️ Troubleshooting

| Error | Penyebab | Fix |
|---|---|---|
| `FileNotFoundError: dataset_sr/...` | Nama folder atau file salah di Drive | Pastikan nama folder `dataset_sr` (case-sensitive) dan nama file persis sama |
| `KeyError: 'job_link'` | Nama kolom di CSV berbeda | Lihat output Cell 3 dan update `COLUMN_MAP` di Cell 4 |
| `CUDA out of memory` | Batch size terlalu besar untuk VRAM GPU | Turunkan `BATCH_SIZE` ke `32` di Cell 4 |
| `Session crashed / RAM habis` | Dataset terlalu besar untuk RAM Colab | Turunkan `MAX_JOB_SAMPLE` ke `50_000` di Cell 4 |
| Encoding crash di tengah jalan | Koneksi terputus / session timeout | Jalankan Cell 8b untuk resume dari checkpoint |
| Download tidak mulai | Browser block popup | Allow popup dari colab.research.google.com, re-run Cell 12 |
| `AssertionError: ALIGNMENT ERROR` | Bug di join/sampling | Jalankan ulang dari Cell 5. Pastikan tidak ada filter yang tidak sengaja |
| Slow encoding tanpa GPU | Runtime type CPU | `Runtime → Change runtime type → T4 GPU` lalu `Runtime → Restart` |
| `Drive not mounted` | Auth expired | Re-run Cell 2, follow authorization link |